# Sprint E10 walkthrough: dynamic risk allocation and loss management

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "efb").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
ALLOC = DATA / "allocation"

In [2]:
# the data hash in the results file must be the hash of the artifacts
# the criteria are read from, recomputed now, not copied
from efb import evaluate

stored = json.loads(
    (ROOT / "sprints" / "E10" / "RESULTS.json").read_text()
)
assert evaluate.e10_data_hash(DATA) == stored["data_hash"], "artifact hash drift"
print("data_hash", stored["data_hash"])
print("verdicts:", stored["reference_values"]["verdicts"])

data_hash c8fd9fc781f03cd0bf7217e4d9c37ecf6512ae76b8983ade8cc10c97d2205cee
verdicts: {'F10.1': 'fail', 'F10.2': 'pass', 'F10.3': 'fail'}


## 1. Every criterion, its stored number and its verdict

In [3]:
for name, block in stored["criteria"].items():
    print(name, block["verdict"], json.dumps(block["stored_numbers"])[:160])

F10.1 fail {"simulated_median_drawdown": -0.14030834893682936, "analytical_median_drawdown": 0.0354329710482708, "relative_gap_at_median": 4.959824558479318, "n_bootstrap"
F10.2 pass {"control_mean_sharpe_diff": -0.487285170300088, "control_improves_sharpe": false, "seed_ew_real_sharpe_diff": 0.0, "seed_mom_ls_real_sharpe_diff": 0.2259660524
F10.3 fail {"raw_dispersion": 0.2663475246430778, "targeted_dispersion": 0.2338401979010474, "dispersion_reduction": 0.12204854085125161, "n_years": 15}


## 2. The design book and the Kelly analysis

The (rho, phi) whose net annualized Sharpe is closest to 1.0, and the Kelly fraction f* = mu / sigma^2 with the growth curve.

In [4]:
from efb import allocate

config = allocate.pick_design_config(DATA)
print("design config", config)
kelly = pd.read_parquet(ALLOC / 'kelly.parquet').iloc[0]
full = kelly['mean_ann'] / kelly['vol_ann'] ** 2
assert abs(full - kelly['kelly_full']) < 1e-9
print('full Kelly', round(kelly['kelly_full'], 4))
print('half Kelly', round(kelly['kelly_half'], 4))
print('growth at full Kelly', round(kelly['growth_full'], 4))
print('growth at half Kelly', round(kelly['growth_half'], 4))
print('growth loss overbet', round(kelly['growth_loss_overbet'], 4))

design config {'rho': 0.02, 'phi': 0.95, 'net_sharpe': 1.1738437142793923, 'gross_sharpe': 1.4561053319301573, 'distance': 0.17384371427939227, 'seed': 1, 'net_sharpe_seed': np.float64(0.9781104435409355)}
full Kelly 9.7811
half Kelly 4.8906
growth at full Kelly 0.4784
growth at half Kelly 0.3588
growth loss overbet 0.0359


## 3. F10.1: the drawdown distribution against the analytical median

In [5]:
drawdown = pd.read_parquet(ALLOC / 'drawdown.parquet').iloc[0]
# the analytical median is ln(2) sigma^2 / (2 mu), recomputed by hand
analytical = np.log(2.0) * kelly['vol_ann'] ** 2 / (2.0 * kelly['mean_ann'])
assert abs(analytical - drawdown['analytical_median_drawdown']) < 1e-9
print('simulated median', round(drawdown['simulated_median_drawdown'], 4))
print('analytical median', round(drawdown['analytical_median_drawdown'], 4))
print('relative gap', round(drawdown['relative_gap_at_median'], 4))

simulated median -0.1403
analytical median 0.0354
relative gap 4.9598


## 4. F10.3: vol targeting and the realized-vol dispersion

In [6]:
voltarget = pd.read_parquet(ALLOC / 'voltarget.parquet').iloc[0]
print('raw dispersion', round(voltarget['raw_dispersion'], 4))
print('targeted dispersion', round(voltarget['targeted_dispersion'], 4))
print('reduction', round(voltarget['dispersion_reduction'], 4))

raw dispersion 0.2663
targeted dispersion 0.2338
reduction 0.122


## 5. F10.2: the stop-loss and its i.i.d. control

In [7]:
stoploss = pd.read_parquet(ALLOC / 'stoploss.parquet')
control = stoploss[stoploss['book'] == 'design'].iloc[0]
print(
    'control mean Sharpe diff',
    round(control['control_mean_sharpe_diff'], 4),
)
assert not bool(control['control_improves_sharpe']), 'the i.i.d. control must not improve'
print(stoploss.to_string(index=False))

control mean Sharpe diff -0.4873
       book  base_sharpe  control_mean_sharpe_diff  control_improves_sharpe  real_book_sharpe_diff  n_bootstrap
     design     0.978110                 -0.487285                    False              -0.394374         2000
    seed_ew     2.207336                       NaN                    False               0.000000            0
seed_mom_ls    -0.247915                       NaN                    False               0.225966            0


## 6. Drawdowns by VIX regime, the risk budget per regime

In [8]:
regime = pd.read_parquet(ALLOC / 'regime.parquet')
print(regime.to_string(index=False))
assert set(regime['vix_tercile']) == {0, 1, 2}

 vix_tercile  mean_vix  max_drawdown  n_underwater  n_obs
           0 12.808793     -0.096651            48     58
           1 16.437241     -0.130857            38     58
           2 24.314828     -0.063602            28     58


## 7. The D9 panel map: which parquet column each panel reads

In [9]:
from dashboard.tabs import d09_risk_allocation as d9

panels = {
    'kelly': d9.kelly_panel(),
    'drawdown': d9.drawdown_panel(),
    'voltarget': d9.voltarget_panel(),
    'stoploss': d9.stoploss_panel(),
    'regime': d9.regime_panel(),
}
for name, panel in panels.items():
    assert not panel.empty, name
    print(name, list(panel.columns))

kelly ['Sharpe (annualized)', 'SE of Sharpe', 'mean (annualized)', 'vol (annualized)', 'full Kelly leverage', 'half Kelly leverage', 'growth at full Kelly', 'growth at half Kelly', 'growth loss when SR overstated by one SE']
drawdown ['simulated_median_drawdown', 'analytical_median_drawdown', 'relative_gap_at_median', 'n_bootstrap', 'n_obs']
voltarget ['raw_dispersion', 'targeted_dispersion', 'dispersion_reduction', 'raw_mean_vol', 'targeted_mean_vol', 'n_years']
stoploss ['book', 'base_sharpe', 'control_mean_sharpe_diff', 'control_improves_sharpe', 'real_book_sharpe_diff', 'n_bootstrap']
regime ['vix_tercile', 'mean_vix', 'max_drawdown', 'n_underwater', 'n_obs']


In [10]:
# the memo cites every criterion and the falsification section
memo = (ROOT / "docs" / "research" / "E10_risk_policy.md").read_text()
joined = " ".join(memo.split())
for name in ("F10.1", "F10.2", "F10.3"):
    assert name in joined, name
assert "What would falsify this?" in joined
assert "synthetic" in joined
print("memo cites every criterion and the falsification section")

memo cites every criterion and the falsification section


In [11]:
# closing checklist: every criterion name is covered by the code
import json as _json

source = "\n".join(
    "".join(cell["source"])
    for cell in _json.loads(
        (ROOT / "notebooks" / "E10_walkthrough.ipynb").read_text()
    )["cells"]
    if cell["cell_type"] == "code"
)
assert all(
    name in source for name in ("F10.1", "F10.2", "F10.3")
)
print("closing checklist: clean")

closing checklist: clean
